# Ансамбль: 2 классификатора + 2 YOLO

- **Классификаторы:** ConvNeXt V2 Tiny (v1, 512) и ConvNeXt V2 Base (v2, 640). Вероятность пневмонии — среднее двух моделей.
- **YOLO:** YOLO26m (v1, 512) и YOLO26l (v2.1, 640). Боксы двух моделей объединяются через Weighted Boxes Fusion: пересекающиеся боксы усредняются, бокс, найденный только одной моделью, получает пониженную уверенность.
- Пороги `t2 × conf` подбираются на val v2. Как и в `03_cross_combinations`, это не совсем честно: модели v1 видели часть этих снимков при обучении.
- Результаты сохраняются в `rsna_project_final/ensemble/`. На Kaggle уходят файлы с разным сжатием боксов K: 1.0 и 0.85 в первом запуске, 0.8 и 0.7 во втором (в ячейке ниже — второй запуск). K = 1.0 — без сжатия, это честный результат.

## 0. Установка, Drive, данные

In [1]:
!pip install -q pydicom pylibjpeg pylibjpeg-libjpeg ultralytics timm ensemble-boxes

In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import zipfile
from pathlib import Path
import pandas as pd

project = Path("/content/drive/MyDrive/rsna_project_final")
zip_path = Path("/content/drive/MyDrive/rsna-pneumonia-detection-challenge.zip")
extract_dir = Path("/content/extract_dir")
if not (extract_dir / ".done").exists():
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(extract_dir)
    (extract_dir / ".done").touch()
train_image_path = extract_dir / "stage_2_train_images"

meta = pd.read_csv(project / "meta_classifier_ConvNeXt_V2_Base_with_aug.csv")   # val v2 — для порогов
labels_1024 = pd.read_csv(extract_dir / "stage_2_train_labels.csv").dropna()
gt_1024 = {pid: g[["x", "y", "width", "height"]].to_numpy() for pid, g in labels_1024.groupby("patientId")}

## 1. Препроцессинг и метрика соревнования

In [4]:
import sys
import numpy as np
import cv2
import pydicom

def percentile_clip(data, lower_percentile=0.5, upper_percentile=99.5):
    low_val, high_val = np.percentile(data, [lower_percentile, upper_percentile])
    return np.clip(data, low_val, high_val)

def read_dicom(dicom_path):
    dcm = pydicom.dcmread(dicom_path)
    img = dcm.pixel_array.astype("float32")
    if dcm.PhotometricInterpretation == "MONOCHROME1":
        img = img.max() - img
    img = percentile_clip(img)
    img = (img - np.min(img)) / (np.max(img) - np.min(img) + sys.float_info.epsilon)
    return np.round(img * 255).astype(np.uint8)

def resize_img(img, size=512):
    resized_img = cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
    return resized_img, resized_img.shape[0] / img.shape[0]

In [5]:
THRESHOLDS = np.round(np.arange(0.4, 0.76, 0.05), 2)   # 0.4, 0.45, ..., 0.75

def iou_matrix(a, b):
    """IoU каждого бокса из a с каждым из b. Боксы — (n, 4): x, y, width, height."""
    ax1, ay1, ax2, ay2 = a[:, 0], a[:, 1], a[:, 0] + a[:, 2], a[:, 1] + a[:, 3]
    bx1, by1, bx2, by2 = b[:, 0], b[:, 1], b[:, 0] + b[:, 2], b[:, 1] + b[:, 3]
    iw = np.clip(np.minimum(ax2[:, None], bx2) - np.maximum(ax1[:, None], bx1), 0, None)
    ih = np.clip(np.minimum(ay2[:, None], by2) - np.maximum(ay1[:, None], by1), 0, None)
    inter = iw * ih
    union = (a[:, 2] * a[:, 3])[:, None] + b[:, 2] * b[:, 3] - inter
    return inter / union


def image_ap(gt, pred, scores):
    """AP одного снимка по правилам Kaggle. None — снимок не учитывается."""
    if len(gt) == 0 and len(pred) == 0:
        return None
    if len(gt) == 0 or len(pred) == 0:
        return 0.0
    pred = pred[np.argsort(-scores)] # сначала самые уверенные
    ious = iou_matrix(pred, gt) # (число предсказаний, число настоящих)
    aps = []
    for t in THRESHOLDS:
        matched, tp = set(), 0
        for i in range(len(pred)):
            cand = [j for j in range(len(gt)) if j not in matched and ious[i, j] > t]
            if cand: # совпал с ещё не занятым настоящим боксом
                matched.add(max(cand, key=lambda j: ious[i, j]))
                tp += 1
        fp, fn = len(pred) - tp, len(gt) - tp
        aps.append(tp / (tp + fp + fn))
    return float(np.mean(aps))

## 2. Модели

In [10]:
import timm, torch
from ultralytics import YOLO
from ensemble_boxes import weighted_boxes_fusion
from tqdm import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CLASSIFIERS = [  # (имя timm, размер, веса)
    ("convnextv2_tiny.fcmae_ft_in22k_in1k", 512, project / "classifier_ConvNeXt_V2_Tiny_no_aug" / "classifier_best_for_f1_classifier_ConvNeXt_V2_Tiny_no_aug.pt"),
    ("convnextv2_base.fcmae_ft_in22k_in1k_384", 640,
     project / "classifier_ConvNeXt_V2_Base_with_aug" / "classifier_best_for_f1_ConvNeXt_V2_Base_with_aug.pt"),
]
YOLOS = [  # (размер, веса)
    (512, project / "yolo" / "split_v1_yolo26m" / "weights" / "best.pt"),
    (640, project / "yolo" / "split_v2_yolo26l_2" / "weights" / "best.pt"),
]
WBF_IOU = 0.55   # с каким IoU боксы разных YOLO считаются одним очагом
NAME = "ensemble_2cls_2yolo"
ENS_DIR = project / "ensemble"   # отдельная папка для результатов ансамбля
ENS_DIR.mkdir(exist_ok=True)

cls_models = []
for name, size, weights in CLASSIFIERS:
    m = timm.create_model(name, pretrained=False, num_classes=3, in_chans=1)
    m.load_state_dict(torch.load(weights, map_location=DEVICE))
    cls_models.append((m.to(DEVICE).eval(), size))
yolo_models = [(YOLO(str(weights)), size) for size, weights in YOLOS]

In [11]:
def run_models(path):
    """Средняя p2 классификаторов и объединённые боксы YOLO [x, y, w, h] в координатах исходного снимка."""
    img = read_dicom(path)
    h, w = img.shape

    p2s = []
    for m, size in cls_models:
        small, _ = resize_img(img, size)
        x = torch.from_numpy(small).float().div(255)[None, None]
        with torch.no_grad():
            p2s.append(torch.softmax(m(((x - 0.449) / 0.226).to(DEVICE)), dim=1)[0, 2].item())

    boxes_list, scores_list, labels_list = [], [], []
    for m, size in yolo_models:
        small, _ = resize_img(img, size)
        r = m.predict(np.repeat(small[..., None], 3, axis=2), imgsz=size, conf=0.01, verbose=False)[0]
        boxes_list.append(r.boxes.xyxyn.cpu().numpy())   # координаты 0..1
        scores_list.append(r.boxes.conf.cpu().numpy())
        labels_list.append(np.zeros(len(r.boxes)))

    if sum(len(b) for b in boxes_list) == 0:
        return np.mean(p2s), np.zeros((0, 4)), np.zeros(0)
    xyxy, s, _ = weighted_boxes_fusion(boxes_list, scores_list, labels_list, iou_thr=WBF_IOU, skip_box_thr=0.01)
    xyxy = xyxy * [w, h, w, h]
    return np.mean(p2s), np.c_[xyxy[:, :2], xyxy[:, 2:] - xyxy[:, :2]], s

## 3. Подбор порогов на val

In [12]:
val_ids = meta.loc[meta["split"] == "val", "patientId"]
val_out = {pid: run_models(train_image_path / f"{pid}.dcm") for pid in tqdm(val_ids, desc="val")}

def score(t2, conf):
    aps = []
    for pid, (p2, pred, s) in val_out.items():
        keep = (s >= conf) & (p2 >= t2)
        ap = image_ap(gt_1024.get(pid, np.zeros((0, 4))), pred[keep], s[keep])
        if ap is not None:
            aps.append(ap)
    return np.mean(aps)

score_val, t2, conf = max((score(t, c), t, c)
                          for t in np.arange(0, 0.81, 0.05) for c in np.arange(0.05, 0.81, 0.05))
print(f"{NAME}: prob2 >= {t2:.2f}, conf >= {conf:.2f} | val {score_val:.4f} (не совсем честно)")
print(f"только YOLO (t2 = 0): {max(score(0, c) for c in np.arange(0.05, 0.81, 0.05)):.4f}")

val: 100%|██████████| 5337/5337 [12:44<00:00,  6.98it/s]


ensemble_2cls_2yolo: prob2 >= 0.45, conf >= 0.10 | val 0.2261 (не совсем честно)
только YOLO (t2 = 0): 0.2081


## 4. Kaggle: сжатие боксов с разными K

In [14]:
test_out = {path.stem: run_models(path)
            for path in tqdm(sorted((extract_dir / "stage_2_test_images").glob("*.dcm")), desc="kaggle")}

for K in [0.8, 0.7]:
    rows = []
    for pid, (p2, pred, s) in test_out.items():
        keep = s >= conf
        pred, s_k = pred[keep], s[keep]
        pred = np.c_[pred[:, 0] + pred[:, 2] * (1 - K) / 2, pred[:, 1] + pred[:, 3] * (1 - K) / 2,
                     pred[:, 2] * K, pred[:, 3] * K]
        pred_str = "" if p2 < t2 else " ".join(
            f"{c:.3f} {x:.1f} {y:.1f} {w:.1f} {h:.1f}" for (x, y, w, h), c in zip(pred, s_k))
        rows.append(dict(patientId=pid, PredictionString=pred_str))
    sub = pd.DataFrame(rows)
    sub.to_csv(ENS_DIR / f"submission_{NAME}_K{K}.csv", index=False)
    print(f"K = {K}: снимков с боксами", (sub["PredictionString"] != "").sum(), "из", len(sub))

kaggle: 100%|██████████| 3000/3000 [07:08<00:00,  7.01it/s]


K = 0.8: снимков с боксами 752 из 3000
K = 0.7: снимков с боксами 752 из 3000


**Результат:** t2 = 0.45 · conf = 0.10 · val 0.226 (только YOLO 0.208)

| K | 1.0 (без сжатия) | 0.85 | 0.8 | 0.7 |
|---|---|---|---|---|
| Kaggle Private | 0.154 | **0.175** | 0.165 | 0.126 |
| Kaggle Public | 0.058 | 0.080 | 0.088 | 0.095 |